# 어떤 고객이 통신 서비스를 해지하는가

## 1. 분석 개요

### 분석 주제

공개 데이터인 Telco Customer Churn 데이터셋을 이용해 **통신사 고객의 특성과 서비스 이용 방식이 고객 이탈과 어떤 관련이 있는지** 확인한다. 가입 기간·계약 유형·이용 서비스·월 요금 등을 중심으로 데이터를 살펴보고, 분류 분석의 전체 과정인 품질 점검 → 탐색적 분석 → 전처리 → 모델링 → 성능 평가 → 해석을 연습한다.

이 분석에서 `Churn`은 고객이 통신 서비스를 해지했는지를 나타낸다. `Yes`는 서비스를 해지한 고객, `No`는 서비스를 유지한 고객을 뜻한다.

### 분석 질문

- **Q1.** 고객의 가입 기간·계약 유형·이용 서비스·요금 등의 특성 중 **서비스 이탈과 가장 뚜렷한 관련이 있는 요인은 무엇인가?**
- **Q2.** 가입 기간과 월 요금 등의 조건을 고려했을 때, 계약 유형이나 인터넷 서비스 이용 방식에 따라 고객 이탈 가능성에 차이가 나타나는가? 또한 다른 조건을 함께 고려한 후에도 이러한 차이가 나타나는가?
- **Q3.** 여러 분류 모형 중 어떤 모형이 이탈 고객을 가장 효과적으로 식별하는가? 
    - 특히 실제 이탈 고객을 놓치지 않는 성능과 전체적인 예측 성능을 함께 고려했을 때 어떤 모형이 가장 적절한가?

**종속변수:** `Churn` — 고객의 서비스 이탈 여부(`Yes`, `No`) → 이진 분류 문제

**설명변수:** `gender`, `SeniorCitizen`, `Partner`, `Dependents`, `tenure`, `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges`

`customerID`는 고객을 구분하기 위한 식별번호이므로 예측을 위한 설명변수에서는 제외한다.

### 이번 노트북에서 연습할 기법 - 머신러닝 분류

데이터의 구조와 품질을 점검하고, 탐색적 분석을 통해 고객 특성과 서비스 이용 방식에 따른 이탈률 차이를 확인한다. 이후 여러 설명변수를 함께 사용하는 분류 모형을 적용하여 각 고객의 서비스 이탈 여부를 예측하고, 어떤 특성이 이탈 가능성과 관련되는지 살펴본다.

실제로 이탈한 고객을 유지 고객으로 잘못 예측하는 경우를 중요하게 살펴본다. 이러한 고객을 놓치면 통신사가 이탈 방지 활동을 수행할 기회를 잃을 수 있기 때문이다.

구체적인 결측값 처리, 범주형 변수 변환, 클래스 불균형 처리 및 모형 구성 방법은 데이터의 실제 분포와 탐색적 분석 결과를 확인한 뒤 결정한다.

> 통계적 유의성 판단 기준: 본 분석에서는 유의수준 (α = 0.05)를 기준으로 한다. 
> 따라서 p-value < 0.05이면 귀무가설을 기각하고 통계적으로 유의한 차이 또는 관계가 있다고 판단하며, p-value ≥ 0.05이면 귀무가설을 기각하지 않는다.

### 1-1. 분석 환경 설정

> 분석에 필요한 개별 함수 **import**

In [1]:
import sys

portfolio_path = r"C:\Data-Analysis-Portfolio"

if portfolio_path not in sys.path:
    sys.path.insert(0, portfolio_path)

from jihyelab import upload_data

In [2]:
from pandas import DataFrame, read_csv
from helpers import *
from IPython.display import display, Markdown

import os
import pandas as pd
import numpy as np
import statsmodels.api as sm

---
## 2. 데이터 소개

### 2-1. 데이터 로딩

In [4]:
# 캐글데이터 주소 모음집
churn = upload_data(
    "churn",
    local=r"C:\Data-Analysis-Portfolio\dataset_repository",
)

churn.head()

📚 통신사 고객의 인구통계·가입 기간·계약 방식·이용 서비스·요금 정보를 바탕으로 고객의 서비스 이탈 여부를 분석하고 예측하는 분류 데이터셋 (출처: https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

    field             description
--  ----------------  ---------------------------------------------------
 0  customerID        고객 식별번호
 1  gender            고객 성별 (Female, Male)
 2  SeniorCitizen     고령자 여부 (고령자=1, 비고령자=0)
 3  Partner           배우자 보유 여부 (Yes, No)
 4  Dependents        부양가족 보유 여부 (Yes, No)
 5  tenure            서비스 가입 기간(개월)
 6  PhoneService      전화 서비스 이용 여부 (Yes, No)
 7  MultipleLines     다중 전화회선 이용 여부 (Yes, No, No phone service)
 8  InternetService   인터넷 서비스 유형 (DSL, Fiber optic, No)
 9  OnlineSecurity    온라인 보안 서비스 이용 여부
10  OnlineBackup      온라인 백업 서비스 이용 여부
11  DeviceProtection  기기 보호 서비스 이용 여부
12  TechSupport       기술 지원 서비스 이용 여부
13  StreamingTV       TV 스트리밍 서비스 이용 여부
14  StreamingMovies   영화 스트리밍 서비스 이용 여부
15  Contract          계약 유형 (Month-to-month, One year, Two year)
16  PaperlessBilling  전자 청구서 이용 여부 (Yes,

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
churn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


#### 문자열 공백 제거
- 문자형 변수의 앞뒤 불필요한 공백을 제거한다.
    - 같은 값이 공백 때문에 서로 다른 범주로 인식되는 오류를 방지하기 위해 사용한다.

In [6]:
for col in churn.select_dtypes(include=['object', 'category']).columns:
    churn[col] = churn[col].astype(str).str.strip()

### 2-2. 출처와 분석 단위

- **출처:** Telco Customer Churn - Kaggle 배포본 [`blastchar/telco-customer-churn`](https://www.kaggle.com/datasets/blastchar/telco-customer-churn), IBM에서 제공한 통신사 고객 이탈 예제 데이터
- **수집 기간:** 배포 자료에 구체적인 수집 연도와 기간이 명시되어 있지 않음
- **분석 단위:** 1행 = 통신사 고객 1명의 인구통계 특성·가입 기간·계약 방식·이용 서비스·요금 정보와 해당 고객의 서비스 이탈 여부(`Churn`) 기록

### 2-3. 변수 정의서

| 변수명 | 의미 | 역할 | 척도 | 단위·범주 |
|---|---|---|---|---|
| `customerID` | 고객을 구분하기 위한 고유번호 | 식별변수 | 명목형 | 고객별 고유 문자열 |
| `gender` | 고객의 성별 | 설명변수 | 명목형(2집단) | `Female`=여성 / `Male`=남성 |
| `SeniorCitizen` | 고령자 여부 | 설명변수 | 명목형(2집단) | `1`=고령자 / `0`=비고령자 |
| `Partner` | 배우자 보유 여부 | 설명변수 | 명목형(2집단) | `Yes`=있음 / `No`=없음 |
| `Dependents` | 부양가족 보유 여부 | 설명변수 | 명목형(2집단) | `Yes`=있음 / `No`=없음 |
| `tenure` | 현재 통신사 서비스 가입 기간 | 설명변수 | 이산형 | 개월 |
| `PhoneService` | 전화 서비스 이용 여부 | 설명변수 | 명목형(2집단) | `Yes`=이용 / `No`=미이용 |
| `MultipleLines` | 다중 전화회선 이용 여부 | 설명변수 | 명목형(3집단) | `Yes`=이용 / `No`=미이용 / `No phone service`=전화 서비스 미이용 |
| `InternetService` | 인터넷 서비스 유형 | 설명변수 | 명목형(3집단) | `DSL` / `Fiber optic` / `No`=인터넷 미이용 |
| `OnlineSecurity` | 온라인 보안 서비스 이용 여부 | 설명변수 | 명목형(3집단) | `Yes`=이용 / `No`=미이용 / `No internet service`=인터넷 미이용 |
| `OnlineBackup` | 온라인 백업 서비스 이용 여부 | 설명변수 | 명목형(3집단) | `Yes`=이용 / `No`=미이용 / `No internet service`=인터넷 미이용 |
| `DeviceProtection` | 기기 보호 서비스 이용 여부 | 설명변수 | 명목형(3집단) | `Yes`=이용 / `No`=미이용 / `No internet service`=인터넷 미이용 |
| `TechSupport` | 기술 지원 서비스 이용 여부 | 설명변수 | 명목형(3집단) | `Yes`=이용 / `No`=미이용 / `No internet service`=인터넷 미이용 |
| `StreamingTV` | TV 스트리밍 서비스 이용 여부 | 설명변수 | 명목형(3집단) | `Yes`=이용 / `No`=미이용 / `No internet service`=인터넷 미이용 |
| `StreamingMovies` | 영화 스트리밍 서비스 이용 여부 | 설명변수 | 명목형(3집단) | `Yes`=이용 / `No`=미이용 / `No internet service`=인터넷 미이용 |
| `Contract` | 서비스 계약 유형 | 설명변수 | 명목형(3집단) | `Month-to-month`=월 단위 / `One year`=1년 / `Two year`=2년 |
| `PaperlessBilling` | 전자 청구서 이용 여부 | 설명변수 | 명목형(2집단) | `Yes`=이용 / `No`=미이용 |
| `PaymentMethod` | 요금 결제 방법 | 설명변수 | 명목형(4집단) | 전자수표 / 우편수표 / 은행 자동이체 / 신용카드 자동이체 |
| `MonthlyCharges` | 고객에게 청구되는 월 이용 요금 | 설명변수 | 연속형 | 금액 |
| `TotalCharges` | 가입 기간 전체의 누적 이용 요금 | 설명변수 | 연속형 | 금액 |
| `Churn` | 고객의 서비스 이탈 여부 | 종속변수 | 명목형(2집단) | `Yes`=이탈 / `No`=유지 |

※ 전체 7,043행, 21개 변수로 구성되어 있다. `Churn`은 고객의 이탈 여부를 나타내는 종속변수이며, 이 분석은 고객의 이탈과 유지를 구분하는 이진 분류 문제에 해당한다.

※ `customerID`는 고객을 구분하기 위한 식별번호로, 고객의 특성이나 서비스 이용 행태를 나타내지 않으므로 분석에서는 제외한다.

※ `SeniorCitizen`은 정수형으로 저장되어 있지만 수치의 크기나 간격을 의미하지 않으므로 연속적인 수치 변수가 아닌 범주형 변수로 취급한다.

※ `TotalCharges`는 현재 문자형(`object`)으로 저장되어 있다. 이는 일부 값에 공백 문자열이 포함되어 있기 때문일 가능성이 있으므로, 공백을 결측값으로 바꾼 후 숫자형으로 변환하여 사용한다.

※ `gender`, `Partner`, `Dependents`, 서비스 이용 여부, 계약 유형, 결제 방법 등의 범주형 변수는 모형에 맞는 형태로 변환한다. `Churn`은 모델링 단계에서 `Yes=1`, `No=0`으로 변환한다.

### 2-4 초기 가설
- **H1. 가입 기간(tenure)이 짧은 고객일수록 서비스 이탈률이 높을 것이다.**
- **H2. 월 이용 요금(MonthlyCharges)이 높은 고객일수록 서비스 이탈 가능성이 높을 것이다.**
- **H3. 온라인 보안(OnlineSecurity)이나 기술 자원(TechSupport)과 같은 부가서비스를 이용하지 않는 고객은 해당 서비스를 이용하는 고객보다 이탈률이 높을 것이다.**

※ 위 가설은 분석 전의 예상이며, 실제 데이터의 분포와 탐색적 분석, 통계적 검정 및 분류 모형의 결과에 따라 기각하거나 지지한다. 또한 개별 변수의 단순한 이탈률 차이뿐만 아니라 여러 고객 특성을 함께 고려한 이후에도 해당 관계가 유지되는지 확인한다.

### 2-5 데이터의 한계
- **인과관계를 확인할 수 없다.** 고객의 특성과 계약 방식, 서비스 이용 여부와 이탈 간의 관련성을 분석하는 자료이므로, 특정 요인이 고객의 서비스 해지를 직접 유발했다고 단정할 수 없다.
- **고객 이탈의 구체적인 이유가 포함되어 있지 않다.** 서비스 품질에 대한 불만, 경쟁사 이동, 고객센터 이용 경험, 통신 장애, 프로모션 및 할인 혜택 등의 정보가 없어 고객이 실제로 서비스를 해지한 원인을 완전히 설명하기 어렵다.
- **고객의 시간에 따른 행동 변화를 확인하기 어렵다.** 현재 자료는 고객별 가입 기간과 누적 요금 드으이 정보는 포함하지만, 월별 요금 변화나 서비스 추가 · 해지 과정과 같은 시계열 이용 기록이 없어 이탈 직전의 행동 변화를 직접 분석할 수 없다.
- **예측 모형의 결과가 실제 이탈 방지 효과를 의미하지는 않는다.** 이 분석은 고객 특성과 서비스 이용 정보를 이용하여 이탈 가능성이 높은 고객을 식별하는 분류 문제를 연습하기 위한 것으로, 모형이 이탈 고객을 정확히 예측하더라도 실제 마케팅이나 고객 유지 활동을 통해 이탈을 방지할 수 있는지는 별도의 검증이 필요하다.